In [1]:
pip install nltk scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# ASSIGNMENT 4: SIMILARITY / PLAGIARISM DETECTOR
# NLP - Skill
# ============================================================

import os
import re
import nltk
import pandas as pd

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------
# 1. Download required NLTK data
# ------------------------------------------------------------

try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

# Similarity threshold for potential plagiarism
PLAGIARISM_THRESHOLD = 70.0

# Folder containing assignment text files
ASSIGNMENT_FOLDER = "assignments"


# ------------------------------------------------------------
# 3. Text Cleaning and Normalization
# ------------------------------------------------------------

def clean_text(text):
    """
    Clean and normalize assignment text.
    """

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)


# ------------------------------------------------------------
# 4. Load Assignment Documents
# ------------------------------------------------------------

def load_assignments(folder):
    """
    Load all .txt assignment files from the given folder.
    """

    documents = {}
    
    if not os.path.exists(folder):
        print(f"\nFolder '{folder}' not found.")
        return documents

    for filename in os.listdir(folder):

        if filename.lower().endswith(".txt"):

            filepath = os.path.join(folder, filename)

            try:
                with open(filepath, "r", encoding="utf-8") as file:
                    text = file.read()

                if text.strip():
                    documents[filename] = text

            except Exception as e:
                print(f"Error reading {filename}: {e}")

    return documents


# ------------------------------------------------------------
# 5. Calculate TF-IDF
# ------------------------------------------------------------

def create_tfidf_vectors(cleaned_documents):

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )

    document_names = list(cleaned_documents.keys())

    document_texts = [
        cleaned_documents[name]
        for name in document_names
    ]

    tfidf_matrix = vectorizer.fit_transform(document_texts)

    return document_names, tfidf_matrix


# ------------------------------------------------------------
# 6. Calculate Cosine Similarity
# ------------------------------------------------------------

def calculate_similarity(tfidf_matrix):

    similarity_matrix = cosine_similarity(tfidf_matrix)

    return similarity_matrix


# ------------------------------------------------------------
# 7. Generate Ranked Similarity Report
# ------------------------------------------------------------

def generate_report(document_names, similarity_matrix):

    results = []

    number_of_documents = len(document_names)

    for i in range(number_of_documents):

        for j in range(i + 1, number_of_documents):

            similarity_score = similarity_matrix[i][j] * 100

            if similarity_score >= PLAGIARISM_THRESHOLD:
                status = "Possible Plagiarism"
            elif similarity_score >= 40:
                status = "Moderate Similarity"
            else:
                status = "Low Similarity"

            results.append({
                "Document 1": document_names[i],
                "Document 2": document_names[j],
                "Similarity (%)": round(similarity_score, 2),
                "Status": status
            })

    # Sort by similarity score
    results = sorted(
        results,
        key=lambda x: x["Similarity (%)"],
        reverse=True
    )

    return results


# ------------------------------------------------------------
# 8. Display Similarity Matrix
# ------------------------------------------------------------

def display_similarity_matrix(document_names, similarity_matrix):

    matrix = pd.DataFrame(
        similarity_matrix * 100,
        index=document_names,
        columns=document_names
    )

    matrix = matrix.round(2)

    print("\n")
    print("=" * 70)
    print("SIMILARITY MATRIX")
    print("=" * 70)

    print(matrix.to_string())

    return matrix


# ------------------------------------------------------------
# 9. Display Ranked Suspicious Pairs
# ------------------------------------------------------------

def display_ranked_report(results):

    print("\n")
    print("=" * 90)
    print("RANKED SIMILARITY / PLAGIARISM REPORT")
    print("=" * 90)

    if not results:
        print("No document pairs available.")
        return

    for rank, result in enumerate(results, start=1):

        print(f"\nRank {rank}")
        print(f"Document 1 : {result['Document 1']}")
        print(f"Document 2 : {result['Document 2']}")
        print(f"Similarity : {result['Similarity (%)']}%")
        print(f"Status     : {result['Status']}")
        print("-" * 60)


# ------------------------------------------------------------
# 10. Main Program
# ------------------------------------------------------------

def main():

    print("=" * 70)
    print("        NLP SIMILARITY / PLAGIARISM DETECTOR")
    print("=" * 70)

    print("\nLoading assignment documents...")

    documents = load_assignments(ASSIGNMENT_FOLDER)

    # Check number of documents
    if len(documents) < 2:

        print("\nERROR: At least two assignment files are required.")

        print("\nCreate a folder named:")
        print("assignments")

        print("\nThen add files such as:")
        print("Assignment_A.txt")
        print("Assignment_B.txt")
        print("Assignment_C.txt")

        return

    print(f"\nTotal assignments loaded: {len(documents)}")

    # --------------------------------------------------------
    # Clean documents
    # --------------------------------------------------------

    print("\nCleaning and normalizing documents...")

    cleaned_documents = {}

    for filename, text in documents.items():

        cleaned_documents[filename] = clean_text(text)

    print("Text cleaning completed.")

    # --------------------------------------------------------
    # Display token information
    # --------------------------------------------------------

    print("\nDocument Information")
    print("-" * 70)

    for filename, text in cleaned_documents.items():

        tokens = text.split()

        print(f"{filename}")
        print(f"Number of tokens: {len(tokens)}")

    # --------------------------------------------------------
    # TF-IDF
    # --------------------------------------------------------

    print("\nCreating TF-IDF vectors...")

    document_names, tfidf_matrix = create_tfidf_vectors(
        cleaned_documents
    )

    print("TF-IDF vectorization completed.")

    # --------------------------------------------------------
    # Cosine Similarity
    # --------------------------------------------------------

    print("\nCalculating cosine similarity...")

    similarity_matrix = calculate_similarity(tfidf_matrix)

    print("Cosine similarity calculation completed.")

    # --------------------------------------------------------
    # Display Matrix
    # --------------------------------------------------------

    matrix = display_similarity_matrix(
        document_names,
        similarity_matrix
    )

    # --------------------------------------------------------
    # Generate Report
    # --------------------------------------------------------

    results = generate_report(
        document_names,
        similarity_matrix
    )

    # --------------------------------------------------------
    # Display Ranked Results
    # --------------------------------------------------------

    display_ranked_report(results)

    # --------------------------------------------------------
    # Save similarity matrix
    # --------------------------------------------------------

    matrix.to_csv("similarity_matrix.csv")

    # --------------------------------------------------------
    # Save plagiarism report
    # --------------------------------------------------------

    report_df = pd.DataFrame(results)

    report_df.to_csv(
        "plagiarism_report.csv",
        index=False
    )

    # --------------------------------------------------------
    # Final Summary
    # --------------------------------------------------------

    suspicious_pairs = [
        result
        for result in results
        if result["Similarity (%)"] >= PLAGIARISM_THRESHOLD
    ]

    print("\n")
    print("=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)

    print(f"Total assignments : {len(documents)}")
    print(f"Pairs compared    : {len(results)}")
    print(f"Threshold         : {PLAGIARISM_THRESHOLD}%")
    print(f"Suspicious pairs  : {len(suspicious_pairs)}")

    if suspicious_pairs:

        print("\nPotentially copied documents:")

        for result in suspicious_pairs:

            print(
                f"- {result['Document 1']} <--> "
                f"{result['Document 2']} : "
                f"{result['Similarity (%)']}%"
            )

    else:

        print("\nNo potentially plagiarized pairs detected.")

    print("\nReports generated:")
    print("1. similarity_matrix.csv")
    print("2. plagiarism_report.csv")

    print("\n")
    print("=" * 70)
    print("        PLAGIARISM DETECTION COMPLETED")
    print("=" * 70)


# ------------------------------------------------------------
# Run Program
# ------------------------------------------------------------

if __name__ == "__main__":
    main()

        NLP SIMILARITY / PLAGIARISM DETECTOR

Loading assignment documents...

Total assignments loaded: 4

Cleaning and normalizing documents...
Text cleaning completed.

Document Information
----------------------------------------------------------------------
Assignment_A.txt
Number of tokens: 35
Assignment_B.txt
Number of tokens: 36
Assignment_C.txt
Number of tokens: 32
Assignment_D.txt
Number of tokens: 29

Creating TF-IDF vectors...
TF-IDF vectorization completed.

Calculating cosine similarity...
Cosine similarity calculation completed.


SIMILARITY MATRIX
                  Assignment_A.txt  Assignment_B.txt  Assignment_C.txt  Assignment_D.txt
Assignment_A.txt            100.00             84.71              4.78              0.42
Assignment_B.txt             84.71            100.00              4.72              0.42
Assignment_C.txt              4.78              4.72            100.00              4.67
Assignment_D.txt              0.42              0.42              4.67   